# Safety Stock Analysis: Assignments 1 and 2
__*Group 3: Fernando Green, Tyler Hensel, Jacob Ramer, Elise Wallace*__

This notebook combines Assignment 1 (data cleaning and exploration) and Assignment 2 (safety stock calculation) for finished goods that are made to stock.

**Contents**
- Setup: Packages and Data
- Assignment 1: Data Cleaning and Exploration (Questions 1–5)
- Assignment 2: Safety Stock Calculation (Questions 1–3 and Sidequest)
- References

## Setup: Packages and Data
All packages used across both assignments are imported here. SciPy's `norm.ppf` and `nbinom.ppf` are used in Assignment 2 to find z-scores and negative binomial reorder points, and `mquantiles` is used for the non-parametric (quantile) safety stock in the Sidequest.

In [1]:
import pandas as pd
import numpy as np
from plotnine import *
import scipy.stats as stats
from scipy.stats import norm
from scipy.stats.mstats import mquantiles

In [2]:
transaction_data = pd.read_csv("../data/transaction_data.csv")
meta_data = pd.read_csv("../data/meta_data.csv")

# Assignment 1: Data Cleaning and Exploration

## Goal of the Exercise

It is important to keep the overarching goal of this exercise, which is to calculate the safety stock by SKU, where the inventory type is a finished good (FG) and the stocking type is made-to-stock (MTS).

It is also important to notate the formula for calculating safety stock, which is below:

$$ SS = z \times \sigma_{d} \times \sqrt{LT}$$

Knowing this, we know that the key features of this dataset would include the SKU, inventory type, stocking type, demand/quantity, and lead time.

## Question 1: DataFrame Formatting

As you load and inspect your transaction dataframe, you'll notice the column names don't follow a consistent convention. Make the necessary changes to ensure all column names follow the same convention as the metadata.

### Answer 1:

#### Headers for meta_data
We want to first get the headers for the meta data in order to identify what needs to be corrected or matched. We also want to get a description of the feature/header for our actual transactional dataset.

We want to display the metadata table first.

In [3]:
meta_data

,feature,description
0,sku_number,Eight character stock keeping unit number
1,inventory_type,Types of inventory including finished good - w...
2,stocking_type,Stockign type policy including make to stock -...
3,lead_time,Total lead time in days
4,unit_price,Unit price US dollar value
5,manufacturing_site,Manufacturing site ID starting with country co...
6,division_code,Four character business division ID
7,transaction_date,Date of Transaction in Year-Month-Day format
8,order_quantity,Order quantity - rounded up for convenience


#### Meta_data Analysis

After looking at the table, we see that there are 9 features with the following data types:

| Feature | Data Type | 
| -------- | ---------- |
| sku_number | Qualitative Nominal |
| inventory_type | Qualitative Ordinal |
| stocking_type | Qualitative Nominal |
| lead_time | Quantitative Discrete |
| unit_price | Quantitative Continuous |
| manufacturing_site | Qualitative Nominal |
| division_code | Qualitative Nominal |
| transaction_date | Qualitative Ordinal |
| order_quantity | Quantitative Discrete |

Now, we'll check the transaction_data to see if the headers and the data types match.


In [4]:
transaction_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 711662 entries, 0 to 711661
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   sku_number          711249 non-null  object 
 1   inventory_type      698248 non-null  object 
 2   stocking_type       697858 non-null  object 
 3   leadTime            711662 non-null  int64  
 4   unit_price          711662 non-null  float64
 5   manufacturing_site  573584 non-null  object 
 6   division Code       661808 non-null  object 
 7   transaction_date    711662 non-null  object 
 8   Order-Quantity      711662 non-null  int64  
dtypes: float64(1), int64(2), object(6)
memory usage: 48.9+ MB


#### Cross-Check & Creating Dictionary

Now we see that generally, all of the data type matches. The only one which we may need to perform some transformation on is "transaction_date".
As far as the headers, it looks like "lead_time", "division_code", and "order_quantity" need to be updated. But just in case, let's go ahead and rename the columns by overwriting the existing column names.

In [5]:
transaction_title = ["sku_number", "inventory_type", "stocking_type", "lead_time", "unit_price", "manufacturing_site", "division_code", "transaction_date", "order_quantity"]

In [6]:
transaction_data.columns = transaction_title

#### Validation for new column names

Next, we will look at the transaction data info to ensure that the correct headers were used.

In [7]:
transaction_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 711662 entries, 0 to 711661
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   sku_number          711249 non-null  object 
 1   inventory_type      698248 non-null  object 
 2   stocking_type       697858 non-null  object 
 3   lead_time           711662 non-null  int64  
 4   unit_price          711662 non-null  float64
 5   manufacturing_site  573584 non-null  object 
 6   division_code       661808 non-null  object 
 7   transaction_date    711662 non-null  object 
 8   order_quantity      711662 non-null  int64  
dtypes: float64(1), int64(2), object(6)
memory usage: 48.9+ MB


## Question 2: 
Inspect the numeric features of your dataframe with the describe function. Provide a summary of these features, and in case there are any oddities/anomalies in your dataframe, locate and correct them.  Please include a note on any oddities you find and a rationale for your suggested correction. You can do so by including a text note in a corresponding markdown shell.

In [8]:
transaction_data.describe()

,lead_time,unit_price,order_quantity
count,711662.000000,711662.000000,711662.000000
mean,25.771684,797.880175,79.788738
std,10.704492,375.268727,41.964285
min,14.000000,-1015.534324,-121.000000
25%,28.000000,982.030704,63.000000
50%,28.000000,996.396930,93.000000
75%,28.000000,1004.287522,109.000000
max,2800.000000,1029.941378,198.000000


### Answer 2:

We see that after running the describe function, there are negative numbers in the "unit_price" and "order_quantity". 
We also see a maximum lead time of 2800. We would want to also investigate any lead times which are beyond 3 standard deviations away from the mean.

#### Order Quantity less than 0

We see that we have 36,437 observations that have a negative order quantity. Looking at the inventory type, we see a mix of finished goods (FG), work-in-progress (WIP), and raw materials (RM). 

Typically, these types of transactions would indicate raw materials which were returned to the supplier, items in WIP which perhaps had a production order (but no manufacturing actually started) that was cancelled, or actual finished goods which were returned by the customer, or even identify a quality issue where items in WIP or even FGs had to be returned/canceled. It could also indicate excess materials which were scrapped.

Normally, we would keep these transactions as is, as it is common for materials to continuously move back and forth between the production lines. Since the ultimate goal of this entire exercise it to calculate safety stock, we need to look at the columns/variables that are used in the calculation of safety stock. This would include demand (quantity) and lead time. 

We performed further analysis below (describing the data for quantities less than 0, and for those greater than 0), and compared these against the entire dataset. We concluded that the mean, standard deviation, and lead time would not change by much.

Even if these transactions were returns or cancellations, the organization still manufactured the product and its replacements if a replacement was needed. Thus, we would consider **dropping** these values.

In [9]:
transaction_data.query('order_quantity < 0')

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
0,0CD5B7B2,WIP,MTO,14,42.905172,NaN,CEA3,2023-01-01,-9
4,9D6AC92B,RM,MTS,14,46.998541,NaN,1C8A,2023-01-01,-8
10,D701F1C7,RM,MTO,14,44.999667,CA-14,1C8A,2023-01-01,-16
63,1990512A,RM,MTS,14,51.537381,PL-38,1536,2023-01-01,-17
69,DFFADF85,FG,MTO,14,52.500816,PL-8F,NaN,2023-01-01,-15
...,...,...,...,...,...,...,...,...,...
711604,B3A50BFB,WIP,MTO,14,51.076793,CA-B4,B395,2025-12-30,-10
711610,99E3E090,FG,MTS,14,44.573580,NaN,BE87,2025-12-30,-15
711612,CEA0C29C,RM,ATO,21,200.918635,NaN,D989,2025-12-30,-5
711626,6B0E4EEF,FG,MTO,14,48.578219,PL-8F,1536,2025-12-30,-12


In [10]:
transaction_data.describe()

,lead_time,unit_price,order_quantity
count,711662.000000,711662.000000,711662.000000
mean,25.771684,797.880175,79.788738
std,10.704492,375.268727,41.964285
min,14.000000,-1015.534324,-121.000000
25%,28.000000,982.030704,63.000000
50%,28.000000,996.396930,93.000000
75%,28.000000,1004.287522,109.000000
max,2800.000000,1029.941378,198.000000


In [11]:
transaction_data.query('order_quantity >= 0').describe()

,lead_time,unit_price,order_quantity
count,675225.000000,675225.000000,675225.000000
mean,26.265708,836.183670,84.508756
std,10.723942,345.760154,37.664268
min,14.000000,-1015.534324,0.000000
25%,28.000000,986.112631,72.000000
50%,28.000000,997.110922,94.000000
75%,28.000000,1004.808758,110.000000
max,2800.000000,1029.941378,198.000000


In [12]:
transaction_data_clean = transaction_data.query('order_quantity >= 0')

#### Order Quantity is 0
We also want to make sure that our data does not contain any order quantities where it is 0. Including these can often skew both the order quantity mean, and the lead time. If no actual order quantity was placed, we would consider this as an data entry error, or it could also indicate a cancellation. Rather than removing the transaction, it could be well-intended for the company's policy to still keep the transaction record, but update the quantity to 0. Nonetheless, for the purposes of our goal, it should be **omitted**.

In [13]:
transaction_data_clean.query('order_quantity == 0').describe()

,lead_time,unit_price,order_quantity
count,4148.000000,4148.000000,4148.0
mean,16.738910,80.791172,0.0
std,4.753034,55.894166,0.0
min,14.000000,32.909604,0.0
25%,14.000000,48.131864,0.0
50%,14.000000,52.730241,0.0
75%,21.000000,97.850171,0.0
max,28.000000,225.605428,0.0


In [14]:
transaction_data_clean = transaction_data_clean.query('order_quantity != 0')
transaction_data_clean.describe()

,lead_time,unit_price,order_quantity
count,671077.000000,671077.000000,671077.000000
mean,26.324595,840.852833,85.031114
std,10.724257,341.644368,37.188021
min,14.000000,-1015.534324,1.000000
25%,28.000000,986.262413,73.000000
50%,28.000000,997.271402,94.000000
75%,28.000000,1004.881084,110.000000
max,2800.000000,1029.941378,198.000000


#### Unit Price less than 0

For the unit pricing, there could be a plethora of reasons as to why we see a negative unit price. They could be cost adjustments, supplier rebates, or it could also be a typo. Cost adjustments could be related to supplier-related cost adjustments, overhead/labor adjustments, or depending on the ERP system and the type of return, the transaction could appear as a negative unit price with a positive order quantity. Some organizations may want to see supplier rebates or cost adjustments differently than actual returns, to help with differentiating what type of "negative" inventory balance occurred. For example, at Amkor Technology, we tend to keep some transaction line items with a positive quantity but a negative unit price to indicate a credit that was issued to our customer for reconciliation purposes. Cost adjustments of raw materials are also transacted in a similar manner, where the quantity is left intact, but the unit price reflects a negative value to issue a credit to the supplier.

If more context is provided and these are actually errors, we would request corrections on these, or just take the mean or median of the unit price by SKU and inventory type.

However, we run into the same issues as the negative quantities. If the ultimate goal is to calculate the safety stock, any credits or cost adjustments should not be included in the order quantity. If these are truly rebates, reconciliations, or adjustments, we don't want the "duplicate" lead times to be included in our safety stock calculation. **In the grand scheme of things, we only have 10 transactions with a negative unit price. Thus, our suggestion is to REMOVE these transaction rows, as the order quantity and lead time values would not "move the needle" much.**

In [15]:
transaction_data_clean.query('unit_price < 0').describe()

,lead_time,unit_price,order_quantity
count,10.000000,10.000000,10.000000
mean,27.300000,-920.553034,98.600000
std,2.213594,253.867643,25.695655
min,21.000000,-1015.534324,34.000000
25%,28.000000,-1004.778647,92.000000
50%,28.000000,-999.283847,107.000000
75%,28.000000,-995.216109,113.500000
max,28.000000,-198.417515,123.000000


##### Removing Negative Unit Price

In [16]:
transaction_data_clean = transaction_data_clean.query('unit_price >= 0')
transaction_data_clean.describe()

,lead_time,unit_price,order_quantity
count,671067.000000,671067.000000,671067.000000
mean,26.324580,840.879081,85.030912
std,10.724334,341.577978,37.188142
min,14.000000,32.909604,1.000000
25%,28.000000,986.262413,73.000000
50%,28.000000,997.271402,94.000000
75%,28.000000,1004.881084,110.000000
max,2800.000000,1029.941378,198.000000


#### Unit price = 0

Next, we also want to look at anything with a unit price of 0. There is **no data** with a unit price of 0. No further action is needed.

In [17]:
transaction_data_clean.query('unit_price == 0').describe()

,lead_time,unit_price,order_quantity
count,0.0,0.0,0.0
mean,NaN,NaN,NaN
std,NaN,NaN,NaN
min,NaN,NaN,NaN
25%,NaN,NaN,NaN
50%,NaN,NaN,NaN
75%,NaN,NaN,NaN
max,NaN,NaN,NaN


#### Lead time greater than 3 standard deviations from the mean (outliers)

While the describe function from pandas already gave us the values for the mean and standard deviation for lead time, I don't want to manually write down and calculate the number to filter. Thus, I've also loaded numpy to perform the calculation, set 2 variables called "lt_3std" and "lt_mean" and using those variables to apply my filter.

In [18]:
lt_3std = 3*np.std(transaction_data_clean["lead_time"])
lt_mean = np.mean(transaction_data_clean["lead_time"])

In [19]:
transaction_data_clean.query('abs(lead_time - @lt_mean) > @lt_3std').describe()

,lead_time,unit_price,order_quantity
count,10.00000,10.000000,10.000000
mean,2520.00000,814.230902,84.900000
std,590.29183,402.870461,39.478405
min,1400.00000,47.259863,13.000000
25%,2800.00000,995.972464,72.750000
50%,2800.00000,1000.637763,96.000000
75%,2800.00000,1008.420766,114.750000
max,2800.00000,1017.014895,123.000000


##### Checking for any lead times with a "0" value

In [20]:
transaction_data_clean.query('lead_time == 0').describe()

,lead_time,unit_price,order_quantity
count,0.0,0.0,0.0
mean,NaN,NaN,NaN
std,NaN,NaN,NaN
min,NaN,NaN,NaN
25%,NaN,NaN,NaN
50%,NaN,NaN,NaN
75%,NaN,NaN,NaN
max,NaN,NaN,NaN


##### Lead Time Corrections

Based on the lead times that were in the transactional data, it is **extremely** unlikely that any lead time would be 3.5 to 7.7 years. Our intuition would tell us to *omit* or remove these line items, or to replace the lead time with the mean or median of the lead time of the sku_number and inventory type (omitting the outliers).

Considering that we only have 10 transactions where the lead time is greater than 1400 days, we would choose to remove these from our dataset.

*Note: The reason why we would not just look at the sku_number is because one SKU can have several inventory types, depending on how they are classified in each manufacturing process. As we add more "value" or "time" with each process, the value may change*

In [21]:
transaction_data_clean = transaction_data_clean.query('abs(lead_time - @lt_mean) <= @lt_3std')
transaction_data_clean.describe()

,lead_time,unit_price,order_quantity
count,671057.000000,671057.000000,671057.000000
mean,26.287420,840.879478,85.030914
std,4.203842,341.577321,37.188138
min,14.000000,32.909604,1.000000
25%,28.000000,986.262413,73.000000
50%,28.000000,997.271402,94.000000
75%,28.000000,1004.881084,110.000000
max,28.000000,1029.941378,198.000000


## Question 3 — NaN Values
Your transactional dataset includes many missing values. Find and address (in your dataframe) the missing values by column. Again, remember to note the NaN values you found and provide a rationale for your suggested action.

### Answer 3:
We want to go through every column and sum the total numbers of NaN values in each column. Since we also REMOVED some rows from the previous step(s), we would also re-reun the isna().sum() function after removing to get the "corrected" values for the counts.

#### SKU Number
Since the SKU number is generally a **unique** identifier, we would NOT fill in the data. However, since we're only interested in calculating the safety stock by SKU and stocking type, we would ultimately opt to **remove** these rows.

However, it should be noted that it ultimately does not matter if we choose to keep these transaction rows as is, or replacing with a value of "Unknown". When we perform a calculation of the safety stock by SKU and inventory type, we would just have a safety stock calculation, but it does not carry any real meaning.

In [22]:
transaction_data_clean.isna().sum()

sku_number               298
inventory_type         12511
stocking_type          13224
lead_time                  0
unit_price                 0
manufacturing_site    130239
division_code          47037
transaction_date           0
order_quantity             0
dtype: int64

##### Removing NaN values from SKU

In [23]:
transaction_data_clean = transaction_data_clean[transaction_data_clean["sku_number"].notna()]
transaction_data_clean.describe()

,lead_time,unit_price,order_quantity
count,670759.000000,670759.000000,670759.000000
mean,26.289873,841.220363,85.063686
std,4.200576,341.269604,37.163560
min,14.000000,32.909604,1.000000
25%,28.000000,986.262413,73.000000
50%,28.000000,997.271402,94.000000
75%,28.000000,1004.881084,110.000000
max,28.000000,1029.941378,198.000000


#### Inventory Type
Inventory type has 12,511 transactions with missing values. 

However, since the ultimate goal of this exercise is to calculate safety stock by SKU for FG MTS products, there is no way for us to accurately "fill" this information in. We don't want to remove more items than necessary. We will later filter for any transactions with and inventory type of "FG", so we would ultimately leave the transactions alone.

If there is a future request to also calculate the safety stock based on inventory type, we would perform further clean-up measures to fill in the information, or omit the transactions from our dataset.

In [24]:
transaction_data_clean.isna().sum()

sku_number                 0
inventory_type         12511
stocking_type          13224
lead_time                  0
unit_price                 0
manufacturing_site    130188
division_code          47011
transaction_date           0
order_quantity             0
dtype: int64

#### Stocking Type
While stocking type is critical for us to filter and calculate safety stock by FG and MTS stocking types, we cannot accurately "fill" in the NaN values for stocking type, since each SKU can carry mutliple stocking types. Thus, we would opt to to **keep** the dataset as it, with a NaN value. We could also remove these rows, but the less that we touch the original dataset, the better.

If there was another request in the future to further cleanse the data, we would either opt to fill or remove these transaction rows.

In [25]:
transaction_data_clean.isna().sum()

sku_number                 0
inventory_type         12511
stocking_type          13224
lead_time                  0
unit_price                 0
manufacturing_site    130188
division_code          47011
transaction_date           0
order_quantity             0
dtype: int64

#### Lead Time
No NaN values for leadtime. No action needed.

#### Unit Price
No NaN values for unit price. No action needed.

#### Manufacturing Site
Again, this feature has no bearing in terms of calculating the safety stock by SKU and stocking type. Since the NaN values does contain important transactions that does carry weight/bearing on our safety stock calculation goal, we would choose to **keep** the transaction data, but **not** fill in the data.

In [26]:
transaction_data_clean["manufacturing_site"].isna().sum()

np.int64(130188)

In [27]:
transaction_data_clean[transaction_data_clean["manufacturing_site"].isna()]

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
7,84FCC369,FG,MTS,28,977.181212,NaN,D9AC,2023-01-01,90
11,A99A8E8D,FG,MTO,28,994.253207,NaN,1C8A,2023-01-01,93
16,5AC3C06B,FG,MTO,28,1015.070220,NaN,1536,2023-01-01,98
27,989A2064,FG,MTS,21,189.249174,NaN,A786,2023-01-01,38
28,B1906A71,RM,MTO,28,996.940154,NaN,BE87,2023-01-01,105
...,...,...,...,...,...,...,...,...,...
711639,261DBD1E,FG,ATO,28,1009.218861,NaN,CEA3,2025-12-30,76
711646,492AD923,WIP,NaN,28,1001.018361,NaN,CC8E,2025-12-30,110
711652,CEA0C29C,FG,MTO,28,1006.562612,NaN,1429,2025-12-30,67
711654,C8EADE59,FG,MTS,28,1000.570170,NaN,F049,2025-12-30,94


#### Division Code

Again, this feature has no bearing in terms of calculating the safety stock by SKU and stocking type. Since the NaN values does contain important transactions that does carry weight/bearing on our safety stock calculation goal, we would choose to **keep** the transaction data, but **not** fill in the data.

In [28]:
transaction_data_clean["division_code"].isna().sum()

np.int64(47011)

In [29]:
transaction_data_clean[transaction_data_clean["division_code"].isna()]

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
3,9EC6C209,WIP,MTS,28,1009.830241,CA-B4,NaN,2023-01-01,122
6,E4AC6BAC,FG,MTO,28,1002.601466,CA-4C,NaN,2023-01-01,134
40,7D780403,RM,MTS,28,999.779816,PL-B2,NaN,2023-01-01,117
46,A5A40F71,WIP,MTS,28,997.426100,PL-8F,NaN,2023-01-01,117
68,DD969A17,FG,MTS,14,52.179130,PL-B2,NaN,2023-01-01,13
...,...,...,...,...,...,...,...,...,...
711547,CEA0C29C,FG,MTO,28,1006.562612,CA-B4,NaN,2025-12-30,112
711558,A3095E6C,WIP,MTO,28,989.006415,PL-8F,NaN,2025-12-30,107
711613,9A73ED78,FG,NaN,28,993.774518,CA-4C,NaN,2025-12-30,99
711628,0D4A17C7,WIP,MTO,28,1006.687408,CA-B4,NaN,2025-12-30,124


#### Transaction Date
No NaN values for transaction date. No action needed.

In [30]:
transaction_data_clean["transaction_date"].isna().sum()

np.int64(0)

#### Order Quantity
No NaN values for order quantity. No action needed.

In [31]:
transaction_data_clean["order_quantity"].isna().sum()

np.int64(0)

#### Summary of NaN counts for each feature after filtering from Question 2

| Feature | Count of NaN | 
| -------- | ---------- |
| sku_number | 298 |
| inventory_type | 12,511 |
| stocking_type | 13,224 |
| lead_time | 0 |
| unit_price | 0 |
| manufacturing_site | 130,188 |
| division_code | 47,011 |
| transaction_date | 0 |
| order_quantity | 0 |

#### Summary of Decisions for each feature
For SKU number, since it is a unique identifier, we would not fill in the data. Considering the context of our goal to calculate the safety stock by SKU and stocking type, it ultimately does not matter if we keep or remove the rows. We would choose to just keep it for now.

For stocking type and inventory type, these features are needed in order to appropriately filter the data. However, there is no accurate way to "fill" the data in. Thus, we would just leave the rows alone, but we would **not** remove the transactions from the dataset. Our position is that the less that we "touch" the original data, the better.

For manufacturing site and divison code, we would keep the data as is, since this feature is not needed for calculating safety stock by SKU and stocking type. However, we would **NOT** remove the transactions from the dataset, since it does contain rows that have key information for our ultimate goal.

Lead time, unit price, transaction date, and order quantity would require no transformation, filling, or removing of NaN values, since there are **no** NaN values.


## Question 4 — Extracting The Country Values

To apply broader breakdowns of your transactional dataset, it will be helpful to add a country column. Recall that the metadata indicates the country information is somewhat captured in the site column. Create a new country column by using the manufacturing_site information. To do so, you will need the helper function str.startswith() (here is a [link](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.startswith.html) to the official documentation), which is defined for a pandas Series object. As the documentation states, str.startswith() tests if the start of each string element matches a given pattern.

### Answer 4:
From the metadata, we know that the countries that we are interested in are USA, Canada, Poland, and Japan.

#### Find unique values under manufacturing site
Next, we want to create a list of the manufacturing sites. We see that there is a pattern, where PL = Poland, US = USA, CA = Canada, and JP = Japan. Rather than mapping out each unique value, we use this pattern to see if we can use the "start with" function. Ideally, we'd like to just see the values of the first 2 characters of the string to then map it to a dictionary.

In [32]:
transaction_data_clean["manufacturing_site"].unique()

array(['PL-38', 'PL-B2', 'CA-B4', 'CA-4C', nan, 'US-6B', 'PL-8F', 'JP-84',
       'PL-0A', 'PL-A2', 'CA-14', 'JP-96', 'JP-38', 'PL-2D', 'US-2B',
       'US-3A', 'PL-F8', 'JP-F5', 'PL-96', 'JP-67'], dtype=object)

#### Creating a dictionary
In order to add a column based on the values of the manufacturing site, we need to create a dictionary first.

In [33]:
country_map = {"CA" : "Canada",
               "JP" : "Japan",
               "PL" : "Poland",
               "US" : "USA"}

#### Mapping to new country
Creates a new column named "country", we take it from the manufacturing site, look at the string of the first 2 characters,and map it to our dictionary. We learned in class that the str function converts the selected column values into a string expression, and creating a index from ":2" would allow the system to select the manufacturing site, transform it into a string expression, and to grab the first 2 characters of the string. Since Python starts a 0, the index would stop **before** the 2nd character (character 0 and character 1). *(Stack Overflow, 2022)*

In [34]:
transaction_data_clean["country"] = transaction_data_clean["manufacturing_site"].str[:2].map(country_map)

In [35]:
transaction_data_clean.sample(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country
515107,9B54A284,FG,MTO,28,986.322569,PL-8F,B395,2025-03-02,81,Poland
167393,46AD289D,RM,MTS,28,997.739775,JP-67,CEA3,2023-09-14,107,Japan
6238,3F31F15C,FG,MTO,28,991.372643,PL-8F,1536,2023-01-10,106,Poland
370656,123D5CE8,FG,MTS,28,990.922748,CA-B4,BE87,2024-07-24,135,Canada
694193,DC82064D,FG,ATO,28,1004.231196,CA-4C,CB7E,2025-12-04,103,Canada
621456,B9E45FC2,WIP,MTO,28,995.036381,PL-B2,F049,2025-08-13,85,Poland
43621,6AE1385C,FG,MTO,21,176.256843,PL-8F,C04F,2023-03-08,44,Poland
411933,62E9A47E,FG,MTO,28,998.644186,CA-B4,F049,2024-09-25,69,Canada
321771,285D1D8D,FG,MTS,28,1000.289315,PL-B2,1C8A,2024-05-09,58,Poland
244572,285D1D8D,FG,MTS,28,1000.289315,NaN,1C8A,2024-01-11,114,NaN


## Question 5 — Helpful Dimensional Information
It's time to focus on the subset of this dataset we need. Since we ultimately want to calculate safety stock, focus on finished goods designated as "make to stock." Create a filtered subset of your transaction dataset and write appropriate code to answer the following questions (for the filtered data).

+ How many unique SKUs are we working with?
+ How many unique Manufacturing Sites are we working with?
+ How many Divisions are we working with?
+ What are the top and bottom 10 transactions by order quantity?
+ What are the top and bottom 10 transactions by total sales value? Hint: Sales is not a feature in this dataset, so we must calculate it using the existing features.

### Answer 5:
#### Creating a filtered subset
We want to create a new subset called "MTS_data" where we are working with the transactions where the stocking_type is only "MTS" and the inventory_type is "FG".

In [36]:
MTS_data = transaction_data_clean.query('stocking_type == "MTS" and inventory_type == "FG"')
MTS_data

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country
1,FED6CA5E,FG,MTS,28,1019.701380,PL-38,1536,2023-01-01,63,Poland
7,84FCC369,FG,MTS,28,977.181212,NaN,D9AC,2023-01-01,90,NaN
8,7E0B9A17,FG,MTS,28,999.436684,US-6B,BE87,2023-01-01,99,USA
12,8A1F415E,FG,MTS,28,1005.366262,CA-B4,1C8A,2023-01-01,78,Canada
13,90D5E95A,FG,MTS,14,41.499619,PL-8F,1536,2023-01-01,7,Poland
...,...,...,...,...,...,...,...,...,...,...
711636,F916CAAF,FG,MTS,28,1019.992250,JP-67,F049,2025-12-30,95,Japan
711638,7E0B9A17,FG,MTS,28,991.088830,PL-B2,CEA3,2025-12-30,100,Poland
711643,00874431,FG,MTS,28,990.146733,PL-B2,D9AC,2025-12-30,125,Poland
711654,C8EADE59,FG,MTS,28,1000.570170,NaN,F049,2025-12-30,94,NaN


#### Unique SKUs Count
To see the unique number of SKUs in our filtered subset, we want to do a value_counts. This value_counts will also automatically *ignore* NaN entries. We also want to find the LENGTH, so we use the len function.

We have 375 unique SKUs. Since we already "dropped" the NaN values, we only have 375 unique SKUs.

In [37]:
len(MTS_data["sku_number"].value_counts())

375

#### Unique manufacturing sites
We perform the same exercise with the SKUs. There are 19 unique manufacturing sites, 20 if including NaN.

In [38]:
len(MTS_data["manufacturing_site"].value_counts())

19

In [39]:
len(MTS_data["manufacturing_site"].value_counts(dropna=False))

20

#### Unique divisions

There are 54 unique divisions for MTS transactions. 55 if including NaN.

In [40]:
len(MTS_data["division_code"].value_counts())

54

In [41]:
len(MTS_data["division_code"].value_counts(dropna=False))

55

#### Finding top 10 transactions by order quantity

In [42]:
MTS_data.sort_values("order_quantity", ascending = False).head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country
33499,7E0B9A17,FG,MTS,28,991.088830,CA-B4,CEA3,2023-02-21,198,Canada
185631,FDDDA064,FG,MTS,28,1010.068090,JP-38,C04F,2023-10-12,194,Japan
180871,285D1D8D,FG,MTS,28,1000.289315,CA-B4,D9AC,2023-10-05,186,Canada
406049,E045018B,FG,MTS,28,995.605558,PL-0A,D9AC,2024-09-16,184,Poland
393301,578DD826,FG,MTS,28,1019.132513,CA-B4,DC0A,2024-08-27,181,Canada
155365,B876EA18,FG,MTS,28,1002.119781,NaN,B395,2023-08-27,180,NaN
278589,5C704F7E,FG,MTS,28,987.620742,PL-8F,1536,2024-03-04,179,Poland
671624,ACC4CB10,FG,MTS,28,998.841700,PL-B2,CB7E,2025-10-30,178,Poland
642523,7D1D8544,FG,MTS,28,996.942519,JP-38,A786,2025-09-15,178,Japan
641161,5E589032,FG,MTS,28,1006.262150,NaN,CEA3,2025-09-13,178,NaN


#### Finding bottom 10 transactions by order quantity
Since we also may want to see any non-negative transactions to exclude the credits or returns, we may want to opt to exclude looking at those.

In [43]:
MTS_data.query('order_quantity > 0').sort_values("order_quantity").head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country
343658,FDDDA064,FG,MTS,14,54.277874,PL-8F,57A9,2024-06-12,1,Poland
405082,B798C3DD,FG,MTS,21,173.792807,NaN,1536,2024-09-14,1,NaN
417468,F9B8CB39,FG,MTS,14,51.835739,NaN,F049,2024-10-03,1,NaN
417854,5C78C11F,FG,MTS,14,56.861257,CA-4C,D9AC,2024-10-04,1,Canada
333,2FA073AE,FG,MTS,14,49.391035,CA-B4,C04F,2023-01-01,1,Canada
325468,5E589032,FG,MTS,21,208.846161,CA-4C,BE87,2024-05-15,1,Canada
247267,91D8FD9E,FG,MTS,14,47.992674,NaN,CB7E,2024-01-15,1,NaN
319834,AC3ADABF,FG,MTS,21,200.715983,PL-A2,A80E,2024-05-07,1,Poland
465236,F2A4F89E,FG,MTS,14,58.020544,PL-B2,F049,2024-12-15,1,Poland
364876,E21018B4,FG,MTS,28,100.030294,PL-38,D9AC,2024-07-15,1,Poland


#### Adding total sales value column
We add a new column titled "total_sales_value" and multiply the order quantity by the unit price.

In [44]:
MTS_data["total_sales_value"] = MTS_data["order_quantity"]*MTS_data["unit_price"]

C:\Users\ferna\AppData\Local\Temp\ipykernel_23332\848128374.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [45]:
MTS_data

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country,total_sales_value
1,FED6CA5E,FG,MTS,28,1019.701380,PL-38,1536,2023-01-01,63,Poland,64241.186944
7,84FCC369,FG,MTS,28,977.181212,NaN,D9AC,2023-01-01,90,NaN,87946.309077
8,7E0B9A17,FG,MTS,28,999.436684,US-6B,BE87,2023-01-01,99,USA,98944.231693
12,8A1F415E,FG,MTS,28,1005.366262,CA-B4,1C8A,2023-01-01,78,Canada,78418.568415
13,90D5E95A,FG,MTS,14,41.499619,PL-8F,1536,2023-01-01,7,Poland,290.497336
...,...,...,...,...,...,...,...,...,...,...,...
711636,F916CAAF,FG,MTS,28,1019.992250,JP-67,F049,2025-12-30,95,Japan,96899.263785
711638,7E0B9A17,FG,MTS,28,991.088830,PL-B2,CEA3,2025-12-30,100,Poland,99108.882956
711643,00874431,FG,MTS,28,990.146733,PL-B2,D9AC,2025-12-30,125,Poland,123768.341576
711654,C8EADE59,FG,MTS,28,1000.570170,NaN,F049,2025-12-30,94,NaN,94053.595963


#### Top 10 transactions by total sales value
Now that we've added a new column, we do the same sorting functions as above.

In [46]:
MTS_data.sort_values("total_sales_value", ascending = False).head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country,total_sales_value
33499,7E0B9A17,FG,MTS,28,991.088830,CA-B4,CEA3,2023-02-21,198,Canada,196235.588252
185631,FDDDA064,FG,MTS,28,1010.068090,JP-38,C04F,2023-10-12,194,Japan,195953.209533
180871,285D1D8D,FG,MTS,28,1000.289315,CA-B4,D9AC,2023-10-05,186,Canada,186053.812682
393301,578DD826,FG,MTS,28,1019.132513,CA-B4,DC0A,2024-08-27,181,Canada,184462.984874
406049,E045018B,FG,MTS,28,995.605558,PL-0A,D9AC,2024-09-16,184,Poland,183191.422709
155365,B876EA18,FG,MTS,28,1002.119781,NaN,B395,2023-08-27,180,NaN,180381.560506
669165,8ED02F71,FG,MTS,28,1013.010673,JP-67,NaN,2025-10-26,177,Japan,179302.889117
641161,5E589032,FG,MTS,28,1006.262150,NaN,CEA3,2025-09-13,178,NaN,179114.662782
253147,6B527587,FG,MTS,28,1011.589319,PL-38,BE87,2024-01-24,176,Poland,178039.720095
671624,ACC4CB10,FG,MTS,28,998.841700,PL-B2,CB7E,2025-10-30,178,Poland,177793.822599


#### Bottom 10 transactions by total sales value

Although our team made the decision to include negative quantities, we thought it might be best to filter out any negative values, as typically we just want to see what is the lowest sales value. Since these possible indicate credits or returns, we also provided another table with the lowest sales values that are non-negative. Note that we already removed the transactions with a quantity of 0.

In [47]:
MTS_data.sort_values("total_sales_value").head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country,total_sales_value
335738,B76BF308,FG,MTS,14,34.391014,PL-8F,1C8A,2024-05-31,1,Poland,34.391014
183845,B76BF308,FG,MTS,14,34.391014,PL-B2,929A,2023-10-09,1,Poland,34.391014
261559,B76BF308,FG,MTS,14,34.391014,NaN,D9AC,2024-02-06,1,NaN,34.391014
285368,63C4377B,FG,MTS,14,37.043908,PL-8F,C04F,2024-03-15,1,Poland,37.043908
551023,63C4377B,FG,MTS,14,37.043908,PL-B2,1536,2025-04-26,1,Poland,37.043908
98491,56D83D7A,FG,MTS,14,37.669826,NaN,CEA3,2023-06-01,1,NaN,37.669826
638470,56D83D7A,FG,MTS,14,37.669826,PL-B2,C04F,2025-09-09,1,Poland,37.669826
420016,369A4789,FG,MTS,14,38.382045,NaN,BE87,2024-10-07,1,NaN,38.382045
21452,369A4789,FG,MTS,14,38.382045,PL-8F,CDD0,2023-02-02,1,Poland,38.382045
376142,369A4789,FG,MTS,14,38.382045,CA-4C,CB7E,2024-08-01,1,Canada,38.382045


# Assignment 2: Safety Stock Calculation

#### Reloading the Raw Data
The raw transaction data is reloaded here so that Assignment 2's cleaning steps start from the original dataset rather than from the rows already removed in Assignment 1.

In [48]:
transaction_data = pd.read_csv("../data/transaction_data.csv")
meta_data = pd.read_csv("../data/meta_data.csv")

## Question 1 — Required Transformation
Create a transformed dataset to filter for appropriate SKUs that are also aggregated with the following statistics: minimum, maximum, average, median, variance, and standard deviation of quantity, along with average lead time.

### Answer 1:
#### Data Cleansing
Similar to assignment 1, we want to clean the data by updating the headers to match the metadata, correcting the lead times, taking the absolute value of order quantity, dropping any transactions where the order quantity is 0, taking the absolute value of the negative unit prices, and filling in NaN values with an "Unknown" category.

In this case, we only want the data to show where the stocking type is MTS. We will create a new dataframe to only look at transactions where the stocking type is MTS.

In [49]:
transaction_title = ["sku_number", "inventory_type", "stocking_type", "lead_time", "unit_price", "manufacturing_site", "division_code", "transaction_date", "order_quantity"]
transaction_data.columns = transaction_title

In [50]:
lt_3std = 3*np.std(transaction_data["lead_time"])
lt_mean = np.mean(transaction_data["lead_time"])

transaction_data_clean = transaction_data.copy()

outlier_condition = (transaction_data_clean["lead_time"] - lt_mean).abs() > lt_3std

transaction_data_clean.loc[outlier_condition, "lead_time"] = (transaction_data_clean.loc[outlier_condition, "lead_time"] / 100)
transaction_data_clean["order_quantity"] = transaction_data_clean["order_quantity"].abs()
transaction_data_clean = transaction_data_clean.query('order_quantity != 0')
transaction_data_clean["unit_price"] = transaction_data_clean["unit_price"].abs()
transaction_data_clean["sku_number"] = transaction_data_clean["sku_number"].fillna("Unknown")
transaction_data_clean["inventory_type"] = transaction_data_clean["inventory_type"].fillna("Unknown")
transaction_data_clean["stocking_type"] = transaction_data_clean["stocking_type"].fillna("Unknown")
transaction_data_clean["manufacturing_site"] = transaction_data_clean["manufacturing_site"].fillna("Unknown")
transaction_data_clean["division_code"] = transaction_data_clean["division_code"].fillna("Unknown")

transaction_data_clean = transaction_data_clean.query('stocking_type == "MTS"')

transaction_data_clean.describe()

,lead_time,unit_price,order_quantity
count,442022.000000,442022.000000,442022.000000
mean,25.910081,809.087564,81.700386
std,4.609872,367.396815,39.688882
min,14.000000,34.391014,1.000000
25%,28.000000,983.932415,66.000000
50%,28.000000,997.110922,93.000000
75%,28.000000,1004.787082,109.000000
max,28.000000,1026.135082,198.000000


#### Data Transformation
In order to use aggregate the quantity by SKU, we have to create a new dataframe. We will use the "groupby" function, along with the aggregate function. Since we are trying to aggregate multiple summarizations of order_quantity, we will have to call the specific functions we'd like in brackets (Pandas development team, n.d.).

Because of the way we are aggregating the quantity and lead time by SKU, it creates multi-level headers. We will overwrite this by just manually renaming the column headers.

##### Aggregating Order Quantities and Lead Times

In [51]:
transformed_data = (transaction_data_clean
    .groupby("sku_number")
    .aggregate({"order_quantity":["min", "max", "mean", "median", "var", "std"],
                "lead_time":"mean"})
    .reset_index())

transformed_data

sku_number order_quantity                                                  \
                          min  max        mean median          var        std   
0     00478A8F              1  166   80.293206   92.0  1575.097842  39.687502   
1     00874431              1  164   98.524059  100.0   558.627557  23.635303   
2     018AA89B              1   21    8.262500    7.0    32.120095   5.667459   
3     01FAD69F              1   23    8.683544    6.0    41.347290   6.430186   
4     02452E5E              1   87   19.990000   17.0   265.543759  16.295513   
..         ...            ...  ...         ...    ...          ...        ...   
492   FEE42AC0              1   28    8.864583    7.0    40.644627   6.375314   
493   FF42B9D4              1   29   10.714286   10.0    44.154135   6.644858   
494   FF711BAA             18  174  100.761988  101.0   397.543668  19.938497   
495   FF9E44A8              1   21    8.672131    8.0    25.624044   5.062020   
496    Unknown              1   33    8.944444    7.0    37.489828   6.122894   

     lead_time  
          mean  
0    25.714546  
1    27.710120  
2    14.000000  
3    14.000000  
4    19.425000  
..         ...  
492  14.000000  
493  14.000000  
494  28.000000  
495  14.000000  
496  14.000000  

[497 rows x 8 columns]

##### Overwriting Column Names

In [52]:
transformed_data.columns = ["sku_number",
                           "min_order_quantity",
                           "max_order_quantity",
                           "avg_order_quantity",
                           "median_order_quantity",
                           "var_order_quantity",
                           "std_order_quantity",
                           "avg_lead_time"]
transformed_data.sample(5)

,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time
417,D5B4B986,1,31,8.862745,8.0,44.907177,6.701282,14.000000
220,6E78AECB,1,30,10.112500,9.5,36.227690,6.018944,14.000000
165,596ECBE0,29,169,99.459259,100.0,391.732125,19.792224,28.000000
314,9F0369DD,1,162,80.329892,93.0,1558.439007,39.477069,25.703895
338,AADC7D9C,1,32,8.797468,8.0,37.240506,6.102500,14.000000


## Question 2 — Safety Stock Calculation
Use the transformed data from Question 1 to calculate three safety stock values for each SKU for the service level coefficients of 75%, 90%, and 95%. You will need to find the z-score of the given service level coefficients, for which you can use the [ppf](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.rv_continuous.ppf.html) function of SciPy.Stat module of the [SciPy](https://docs.scipy.org/doc/scipy/reference/) package, or simply use the numeric values from any standard normal distribution function.

### Answer 2:

#### Type of Distribution
##### Using Coefficient of Variation Rule for Normal Distribution
First, we want to check to see if our inventory is normally distributed or if a different distribution should be used. We will add another column for the coefficient of variation. For formula for the coefficient of variation is as follows:

$\text{CV} = \frac{\text{Std Dev}}{\text{Mean}}$

If the CV is less thna 30%, demand is relatively stable and predictable, relative to its volume, indicating that we can most likely use a normal distribution (Oracle, n.d.).

Now, we calculate the CV and find the SKUs which we can apply a normal distribution.

In [53]:
transformed_data["cv"] = transformed_data["std_order_quantity"]/transformed_data["avg_order_quantity"]

In [54]:
transformed_data.query('cv <= 0.3')

,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time,cv
1,00874431,1,164,98.524059,100.0,558.627557,23.635303,27.710120,0.239894
8,048B94FE,19,165,99.866627,100.0,418.892640,20.466867,28.000000,0.204942
18,0839933A,27,169,99.800364,100.0,408.627395,20.214534,28.000000,0.202550
25,0D4A17C7,1,161,95.801063,99.0,697.599503,26.412109,27.413223,0.275697
28,123D5CE8,30,161,100.403285,101.0,406.071590,20.151218,28.000000,0.200703
...,...,...,...,...,...,...,...,...,...
479,F633F4BB,36,171,100.611180,100.5,370.224114,19.241209,28.000000,0.191243
485,FA9DDBC6,1,164,95.920863,99.0,736.716325,27.142519,27.356945,0.282968
486,FB796951,1,164,95.587519,99.0,737.135357,27.150237,27.357196,0.284035
491,FED6CA5E,37,165,98.974586,100.0,375.621471,19.380956,28.000000,0.195818


##### Variance to Mean Ratio
Next, we will use a variance to mean ratio to determine whether we the SKUs have a binomial, poisson, or negative binomial distribution, using different "generalized" thresholds (CASRAI, n.d.).

$\text{VMR} = \frac{\text{Variance}}{\text{Mean}}$

In [55]:
transformed_data["VMR"] = transformed_data["var_order_quantity"]/transformed_data["avg_order_quantity"]

##### Filtering for Binomial Distributions
After filtering for any SKUs with a high coefficient of variation and variance to mean ratio much then 1, we identified 0 SKUs which indicate a binomial distribution.

In [56]:
transformed_data.query('cv > 0.3 and VMR < 0.8')

,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time,cv,VMR


##### Filtering for Poisson Distributions
After filtering for any SKUs with a high coefficient of variation and medium variance to mean ratio close to 1, we identified 0 SKUs which indicate a poisson distribution. We selected a ratio between 0.8 and 1.2

In [57]:
transformed_data.query('cv > 0.3 and VMR >= 0.8 and VMR <1.2')

,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time,cv,VMR


##### Filtering for Negative Binomial Distributions
After filtering for any SKUs with a high coefficient of variation and a variance to mean ratio greater than 1, we identified the remaining non-normally distributed SKUs as having a negative binomial distribution. This meeans that most SKUs have a fairly low demand/order quantity but experience erratic or unpredictable high-quantity spikes. Using a safety stock calculation which assumes a normal distribution, we may experience more stockouts on these SKUs.

In [58]:
transformed_data.query('cv > 0.3 and VMR >= 1.2')

,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time,cv,VMR
0,00478A8F,1,166,80.293206,92.0,1575.097842,39.687502,25.714546,0.494282,19.616826
2,018AA89B,1,21,8.262500,7.0,32.120095,5.667459,14.000000,0.685925,3.887455
3,01FAD69F,1,23,8.683544,6.0,41.347290,6.430186,14.000000,0.740502,4.761568
4,02452E5E,1,87,19.990000,17.0,265.543759,16.295513,19.425000,0.815183,13.283830
5,02CFF6A9,1,88,22.865672,20.0,252.631602,15.894389,21.000000,0.695120,11.048510
...,...,...,...,...,...,...,...,...,...,...
490,FE1EE752,1,43,9.906250,8.0,58.965998,7.678932,14.000000,0.775160,5.952404
492,FEE42AC0,1,28,8.864583,7.0,40.644627,6.375314,14.000000,0.719189,4.585058
493,FF42B9D4,1,29,10.714286,10.0,44.154135,6.644858,14.000000,0.620187,4.121053
495,FF9E44A8,1,21,8.672131,8.0,25.624044,5.062020,14.000000,0.583711,2.954757


#### Find z-scores for Percentiles for Normal Distributions

In [59]:
z_75 = norm.ppf(0.75)
z_90 = norm.ppf(0.90)
z_95 = norm.ppf(0.95)

#### Calculating Safety Stock for Normal Distributions
Now we use the following function to calculate the safety stock for each SKU. We will use our transformed dataset to complete this.

$$ SS = z_{\alpha} \times \sigma_{d} \times \sqrt{LT}$$

We also want to display whole numbers for these, so we will use the floor division (//) and remainder (%) functions to achieve this. The floor division will give us the "rounded" integer. We also use the remainder function, along with a logical operator to return a value of 0 or 1. If the remainder > 0, then the result should be 1. If the remainder is 0, then the result of that subcalculation would be 0. We also want to change the data type to "int64".

To demonstrate this, we've provided an example below:

- We select a random value of 17.4
- When we do the floor division, it will return a value of 17.
- When we do the remainder, it should return a value of 4.
- We also have a logical operator for the remainder, saying if the remainder (4) is greater than 0, then return a value of 1.
- In this case, since we have a remainder that is greater than 0, it would return a value of 1.
- We add this boolean response of 1 to the floor division result of 17.
- Our final result would be 18.

##### Safety Stock at 75%

In [60]:
raw_ss_75_norm = z_75 * transformed_data["std_order_quantity"] * np.sqrt(transformed_data["avg_lead_time"])
transformed_data["ss_75_norm"] = (raw_ss_75_norm // 1 + (raw_ss_75_norm % 1 > 0)).astype("int64")
transformed_data.sample(5)

,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time,cv,VMR,ss_75_norm
487,FC7D2F78,1,37,9.776536,8.0,55.095851,7.422658,14.000,0.759232,5.635518,19
128,4526F023,1,32,9.383562,8.0,47.906393,6.921444,14.000,0.737614,5.105353,18
327,A5F321CB,1,176,85.330689,94.0,1401.477396,37.436311,28.000,0.438720,16.424072,134
112,3F10C4FA,1,25,9.105263,8.0,37.028772,6.085127,14.000,0.668309,4.066744,16
336,AAAD8A64,1,162,84.430500,96.0,1507.755047,38.829822,25.578,0.459903,17.857943,133


##### Safety Stock at 90%

In [61]:
raw_ss_90_norm = z_90 * transformed_data["std_order_quantity"] * np.sqrt(transformed_data["avg_lead_time"])
transformed_data["ss_90_norm"] = (raw_ss_90_norm // 1 + (raw_ss_90_norm % 1 > 0)).astype("int64")
transformed_data.sample(5)

,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time,cv,VMR,ss_75_norm,ss_90_norm
21,08E80D67,1,22,8.500000,8.0,34.706667,5.891236,14.000000,0.693087,4.083137,15,29
376,C2D23FE2,1,173,96.128596,99.0,729.438307,27.008116,27.391991,0.280958,7.588151,96,182
147,5039C694,1,30,8.322981,7.0,34.970031,5.913546,14.000000,0.710508,4.201623,15,29
446,E1CF58A6,35,169,100.225412,100.0,399.949769,19.998744,28.000000,0.199538,3.990503,72,136
131,4734F298,1,26,9.093023,8.0,38.226539,6.182761,14.000000,0.679946,4.203942,16,30


##### Safety Stock at 95%

In [62]:
raw_ss_95_norm = z_95 * transformed_data["std_order_quantity"] * np.sqrt(transformed_data["avg_lead_time"])
transformed_data["ss_95_norm"] = (raw_ss_95_norm // 1 + (raw_ss_95_norm % 1 > 0)).astype("int64")
transformed_data.sample(5)

,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time,cv,VMR,ss_75_norm,ss_90_norm,ss_95_norm
102,397C3081,1,41,10.690594,9.5,52.502045,7.245830,19.891089,0.677776,4.911050,22,42,54
197,645F6ADD,42,162,100.040684,100.0,385.742887,19.640338,28.000000,0.196324,3.855860,71,134,171
109,3E350911,1,167,90.363333,96.0,1108.731729,33.297623,26.615556,0.368486,12.269708,116,221,283
309,9B24EB0B,5,167,99.064497,99.0,408.187117,20.203641,28.000000,0.203944,4.120418,73,138,176
431,DC272A30,35,165,99.873950,100.0,408.059777,20.200490,28.000000,0.202260,4.085748,73,137,176


#### Calculating Safety Stock for Negative Binomial Distributions
For the SKUs with a negative binomial distribution, we want to identify the reorder point and average demand during lead time to calculate safety stock. The formula is as follows:
$$\text{Safety Stock} = ROP - \mu_{LT}$$

To obtain the reorder point, we would use the SciPy.stats nbinom.ppf function.
To use the function, we would also need to calculate several other parameters. We would need the shape/dispersion (n), probability parameter (p), average demand during lead time, and average variance during lead time. The formulas are as follows:

$$n = \frac{\mu_{LT} \times p}{1 - p}$$
$$p = \frac{\mu_{LT}}{\sigma^2_{LT}}$$
$$\mu_{LT} = \mu_{\text{daily}} \times \text{Lead Time}$$
$$\sigma^2_{LT} = \sigma^2_{\text{daily}} \times \text{Lead Time}$$

The n parameter (sometimes known as k) and the p-parameter are from a statistical technique called "method of moments" (Zaiontz, n.d.).

In [63]:
transformed_data["mean_during_lt"] = transformed_data["avg_order_quantity"]*transformed_data["avg_lead_time"]
transformed_data["var_during_lt"] = transformed_data["var_order_quantity"]*transformed_data["avg_lead_time"]

In [64]:
transformed_data["p_parameter"] = transformed_data["mean_during_lt"] / transformed_data["var_during_lt"]

In [65]:
transformed_data["n_parameter"] = ((transformed_data["mean_during_lt"] * 
                                    transformed_data["p_parameter"]) / 
                                    (1- transformed_data["p_parameter"]))

In [66]:
transformed_data["rop_75"] = stats.nbinom.ppf(0.75, transformed_data["n_parameter"], transformed_data["p_parameter"])
transformed_data["rop_90"] = stats.nbinom.ppf(0.90, transformed_data["n_parameter"], transformed_data["p_parameter"])
transformed_data["rop_95"] = stats.nbinom.ppf(0.95, transformed_data["n_parameter"], transformed_data["p_parameter"])

##### Adding Safety Stock columns for negative binomial distributions
Now, we substrate the reorder point from the average demand during lead time to obtain the safety stock at 75%, 90%, and 95%.

We would use the same rounding methodology as the normal distribution safety stocks to ensure that we are providing whole values for safety stock quantities.

In [67]:
raw_ss_75_bn = transformed_data["rop_75"] - transformed_data["mean_during_lt"]
raw_ss_90_bn = transformed_data["rop_90"] - transformed_data["mean_during_lt"]
raw_ss_95_bn = transformed_data["rop_95"] - transformed_data["mean_during_lt"]

transformed_data["ss_75_bn"] = (raw_ss_75_bn // 1 + (raw_ss_75_bn % 1 > 0)).astype("int64")
transformed_data["ss_90_bn"] = (raw_ss_90_bn // 1 + (raw_ss_90_bn % 1 > 0)).astype("int64")
transformed_data["ss_95_bn"] = (raw_ss_95_bn // 1 + (raw_ss_95_bn % 1 > 0)).astype("int64")

#### Finalized Safety Stock Column
Since we want to just consolidate the "correct" safety stock to each SKU, we will use the np.where function to call upon the safety stock at 75^, 90%, and 95%, depending on the value of the coefficient of variation. Since we already identified that anything with a coefficient of variation which is greater than 30% is using a negative binomial distribution, we just need to build logic that says:

If the CV is less than or equal to 0.3, then grab the NORMAL safety stock, else use the binomial negative safety stock calculation.

In [68]:
transformed_data["ss_75"] = np.where(transformed_data["cv"] <= 0.3,
                                     transformed_data["ss_75_norm"],
                                     transformed_data["ss_75_bn"])

transformed_data["ss_90"] = np.where(transformed_data["cv"] <= 0.3,
                                     transformed_data["ss_90_norm"],
                                     transformed_data["ss_90_bn"])

transformed_data["ss_95"] = np.where(transformed_data["cv"] <= 0.3,
                                     transformed_data["ss_95_norm"],
                                     transformed_data["ss_95_bn"])

transformed_data.sample(5)

,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time,cv,VMR,...,n_parameter,rop_75,rop_90,rop_95,ss_75_bn,ss_90_bn,ss_95_bn,ss_75,ss_90,ss_95
395,CA656E8B,1,79,20.250000,17.0,227.088235,15.069447,19.589286,0.744170,11.214234,...,38.836299,440.0,484.0,512.0,44,88,116,44,88,116
398,CC473D76,1,165,86.588469,94.0,1254.360643,35.416954,26.764911,0.409026,14.486463,...,171.841398,2438.0,2555.0,2627.0,121,238,310,121,238,310
118,4050F086,13,167,100.563413,100.0,384.419222,19.606612,28.000000,0.194968,3.822655,...,997.562824,2885.0,2949.0,2988.0,70,134,173,70,133,171
364,B9E45FC2,1,50,11.149068,10.0,55.554033,7.453458,21.347826,0.668527,4.982841,...,59.758443,260.0,283.0,297.0,22,45,59,22,45,59
168,5A891AFC,1,82,23.682081,21.0,266.194287,16.315462,21.000000,0.688937,11.240325,...,48.565226,546.0,595.0,626.0,49,98,129,49,98,129


## Question 3 — Safety Stock Distribution
Now that we have the safety stock values, write appropriate code to answer the following questions. For this question, you may focus on the 95% service level coefficient.

- Which SKU has the largest safety stock?
- Which SKU has the smallest safety stock?
- What is the average safety stock value across all SKUs?

### Answer 3:
Before answering questions, now I want to create a copy of my transformed data into a neat and clean table with just the SKU and safety stocks for each method (since this would also be a way to validate the data). We create a new dataframe from the aggregated transformed data with sku number, and the 3 columns for safety stock at 95% with both methods (normal and negative binomial).

In [69]:
selected_columns = ["sku_number", "ss_95_norm", "ss_95_bn", "ss_95"]
data_95_ss = transformed_data[selected_columns].copy()

#### Largest Safety Stock
In order to find the SKU with the largest safety stock, using a 95% service level agreement (SLA), we would need to sort the values by DESCENDING values from the "ss_95" column. We also used the .head function, but wanted to print multiples, just in case there were multiple SKUs with the highest safety stock quantity.

The SKU with the largest safety stock is AC3ADABF, since it would use the negative binomial distribution. However, if assuming all SKUs had a normal distribution, the SKU with the highest safety stock is 089FA02B.

In [70]:
data_95_ss.sort_values("ss_95", ascending = False).head(5)

,sku_number,ss_95_norm,ss_95_bn,ss_95
342,AC3ADABF,352,370,370
20,089FA02B,353,366,366
471,F0610752,351,362,362
489,FDDDA064,349,360,360
156,56D83D7A,348,359,359


In [71]:
data_95_ss.sort_values("ss_95_norm", ascending=False).head(5)

,sku_number,ss_95_norm,ss_95_bn,ss_95
20,089FA02B,353,366,366
342,AC3ADABF,352,370,370
471,F0610752,351,362,362
489,FDDDA064,349,360,360
156,56D83D7A,348,359,359


#### Smallest Safety Stock
In order to find the SKU with the smallest safety stock, using a 95% service level agreement (SLA), we would need to sort the values by ASCENDING values from the "ss_95" column. We also used the .head function, but wanted to print multiples, just in case there were multiple SKUs with the smallest safety stock quantity.

The SKU with the smallest safety stock is 58F77D5E, using both methods.

In [72]:
data_95_ss.sort_values("ss_95", ascending = True).head(5)

,sku_number,ss_95_norm,ss_95_bn,ss_95
163,58F77D5E,29,30,30
61,243ECBFB,31,33,33
495,FF9E44A8,32,33,33
49,1F900448,33,34,34
384,C6D49812,34,35,35


In [73]:
data_95_ss.sort_values("ss_95_norm", ascending = True).head(5)

,sku_number,ss_95_norm,ss_95_bn,ss_95
163,58F77D5E,29,30,30
61,243ECBFB,31,33,33
495,FF9E44A8,32,33,33
49,1F900448,33,34,34
384,C6D49812,34,35,35


#### Average Safety Stock
To obtain the average safety stock, we can just use the .describe() function to find the mean.

The average safety stock is about ~123 units, when doing a mix between normal and negative binomial distributions, ~124 units when looking only at a negative binomial distribution, and ~120 units when looking only at a normal distribution.

In [74]:
data_95_ss.describe()

,ss_95_norm,ss_95_bn,ss_95
count,497.000000,497.000000,497.000000
mean,119.997988,123.674044,123.146881
std,96.063036,97.887579,97.463251
min,29.000000,30.000000,30.000000
25%,41.000000,44.000000,44.000000
50%,66.000000,69.000000,69.000000
75%,176.000000,178.000000,176.000000
max,353.000000,370.000000,370.000000


## SIDEQUEST
While we already performed the above analysis, we will implement the **SUGGESTED** sidequest of calculating non-parametric safety stock using the empirical quantile formula (because one of our team members went DEEP down the rabbit hole and didn't see the side quest until after she already completed the above exercise). We know the following safety stock formula for parametric (normal) distribution:

$$ SS = z_{\alpha} \times \sigma_{d} \times \sqrt{LT}$$

For a non-parametric safety stock calculation, we looking at our dataset, finding the stock quantity at the 75th, 90th and 95th percentile. Rather than using a z score with the standard deviation, we are essentially calculating the nth percentile of our dataset and substracting from the mean to find the **spread**. In other words, the z-score and standard deviation is your *demand buffer*. We are just replacement the demand buffer with a different variation, where we are getting the nth quartile and substracting from the mean.

$$\text{Safety Stock} = (\text{Quantile}_{CSL}(d) - \bar{d}) \times \sqrt{LT}$$

Since the quantiles need aggregated data by SKU, we will create 3 new aggregated series with the quantile calculation at each service level. We also discovered that we had to use the float function, since we needed to transform the array to Python scalars, due to a type error (Hakyimlab, 2026). Then, we will merge this information with the existing "tranformed_data" table (pd.merge, since we already have the SKU_number in both tables), then calculate the safety stock, using the above formula with the appropriate rounding.

In [75]:
q75 = transaction_data_clean.groupby("sku_number")["order_quantity"].aggregate(lambda x: float(mquantiles(x, prob=0.75)[0])).reset_index(name = "q75_order_quantity")
q90 = transaction_data_clean.groupby("sku_number")["order_quantity"].aggregate(lambda x: float(mquantiles(x, prob=0.90)[0])).reset_index(name = "q90_order_quantity")
q95 = transaction_data_clean.groupby("sku_number")["order_quantity"].aggregate(lambda x: float(mquantiles(x, prob=0.95)[0])).reset_index(name = "q95_order_quantity")

In [76]:
transformed_data = pd.merge(transformed_data, q75, on = "sku_number", how = "left")
transformed_data = pd.merge(transformed_data, q90, on = "sku_number", how = "left")
transformed_data = pd.merge(transformed_data, q95, on = "sku_number", how = "left")

In [77]:
raw_ss_np_75 = (transformed_data["q75_order_quantity"] - transformed_data["avg_order_quantity"]) * np.sqrt(transformed_data["avg_lead_time"])
raw_ss_np_90 = (transformed_data["q90_order_quantity"] - transformed_data["avg_order_quantity"]) * np.sqrt(transformed_data["avg_lead_time"])
raw_ss_np_95 = (transformed_data["q95_order_quantity"] - transformed_data["avg_order_quantity"]) * np.sqrt(transformed_data["avg_lead_time"])

In [78]:
transformed_data["ss_np_75"] = (raw_ss_np_75 // 1 + (raw_ss_np_75 % 1 > 0)).astype("int64")
transformed_data["ss_np_90"] = (raw_ss_np_90 // 1 + (raw_ss_np_90 % 1 > 0)).astype("int64")
transformed_data["ss_np_95"] = (raw_ss_np_95 // 1 + (raw_ss_np_95 % 1 > 0)).astype("int64")

In [79]:
transformed_data

,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time,cv,VMR,...,ss_95_bn,ss_75,ss_90,ss_95,q75_order_quantity,q90_order_quantity,q95_order_quantity,ss_np_75,ss_np_90,ss_np_95
0,00478A8F,1,166,80.293206,92.0,1575.097842,39.687502,25.714546,0.494282,19.616826,...,342,133,262,342,108.0,123.00,131.00,141,217,258
1,00874431,1,164,98.524059,100.0,558.627557,23.635303,27.710120,0.239894,5.669961,...,208,84,160,205,114.0,126.00,133.00,82,145,182
2,018AA89B,1,21,8.262500,7.0,32.120095,5.667459,14.000000,0.685925,3.887455,...,37,14,28,37,14.0,17.00,18.00,22,33,37
3,01FAD69F,1,23,8.683544,6.0,41.347290,6.430186,14.000000,0.740502,4.761568,...,42,16,32,42,15.0,18.00,20.64,24,35,45
4,02452E5E,1,87,19.990000,17.0,265.543759,16.295513,19.425000,0.815183,13.283830,...,125,46,95,125,30.0,41.00,52.59,45,93,144
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
492,FEE42AC0,1,28,8.864583,7.0,40.644627,6.375314,14.000000,0.719189,4.585058,...,42,15,31,42,13.0,17.98,20.79,16,35,45
493,FF42B9D4,1,29,10.714286,10.0,44.154135,6.644858,14.000000,0.620187,4.121053,...,43,16,33,43,14.3,20.00,22.00,14,35,43
494,FF711BAA,18,174,100.761988,101.0,397.543668,19.938497,28.000000,0.197877,3.945373,...,176,72,136,174,113.0,126.00,134.00,65,134,176
495,FF9E44A8,1,21,8.672131,8.0,25.624044,5.062020,14.000000,0.583711,2.954757,...,33,13,25,33,11.3,16.96,18.54,10,32,37


### Re-answering Question 3 for non-parametric (quantile) calculation
We will re-select new columns and perform sorting by the largest, smallest, and average safety stock.

When looking at the safety stock using the non-parametric (quantile) safety stock calculation, we see that the SKU is AC3ADABF, which matched our answer for the negative binomial distribution. However, the safety stock is still much lower than our other calculations, and only 3 of the 5 top SKUs appeared from our previous list.

When looking at the smallest safety stock, SKU 58F77D5E is still the smallest across all three methods. However, the value for the non-parametric method resulted in a higher value in comparison to the other 2 methods. Additionally, we again see that only 3 of the 5 bottom SKUs appeared in our non-parametric list, compared to our previous list.

Lastly, the average safety stock is reduced down to ~114 units, which is a much more conservative value in comparison to both methods. If the organization wants to be more conservative with their approach, it may be best to use the non-parametric method for safety stock calculation, the average safety stock is much lower than any of the other methods that we have attempted.

In [80]:
selected_columns = ["sku_number", "ss_95_norm", "ss_95_bn", "ss_95", "ss_np_95"]
data_95_ss = transformed_data[selected_columns].copy()

In [81]:
data_95_ss.sort_values("ss_np_95", ascending = False).head(5)

,sku_number,ss_95_norm,ss_95_bn,ss_95,ss_np_95
342,AC3ADABF,352,370,370,323
166,5978F66E,345,358,358,278
62,248D68DF,330,340,340,266
20,089FA02B,353,366,366,263
471,F0610752,351,362,362,261


In [82]:
data_95_ss.sort_values("ss_np_95", ascending = True).head(5)

,sku_number,ss_95_norm,ss_95_bn,ss_95,ss_np_95
163,58F77D5E,29,30,30,32
61,243ECBFB,31,33,33,35
49,1F900448,33,34,34,35
445,E0C722AA,37,38,38,36
380,C565BADC,40,42,42,36


In [83]:
data_95_ss.describe()

,ss_95_norm,ss_95_bn,ss_95,ss_np_95
count,497.000000,497.000000,497.000000,497.000000
mean,119.997988,123.674044,123.146881,113.981891
std,96.063036,97.887579,97.463251,71.136013
min,29.000000,30.000000,30.000000,32.000000
25%,41.000000,44.000000,44.000000,49.000000
50%,66.000000,69.000000,69.000000,76.000000
75%,176.000000,178.000000,176.000000,178.000000
max,353.000000,370.000000,370.000000,323.000000


## References

CASRAI. (n.d.). Negative binomial regression overdispersion. CASRAI Guides. Retrieved November 12, 2026, from https://casrai.org/guides/negative-binomial-regression-overdispersion

Hakyimlab. (2026, April 19). TypeError: only 0-dimensional arrays can be converted to Python scalars with numpy >= 2.x (Issue No. 225) [Online forum post]. GitHub. https://github.com/hakyimlab/MetaXcan/issues/225

Oracle. (n.d.). NetSuite help center: Inventory management guidelines. Oracle Help Center. Retrieved November 12, 2026, from https://docs.oracle.com/en/cloud/saas/netsuite/ns-online-help/article_0702084957.html#subsect_75201401147

Pandas development team. (n.d.). Pandas.DataFrame.aggregate. pandas. Retrieved November 7, 2026, from https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.aggregate.html

Stack Overflow. (2022, March 16). How to substring values from a pandas df [Online forum post]. Stack Exchange Inc. https://stackoverflow.com/questions/71537795/how-to-substring-values-from-a-pandas-df

Zaiontz, C. (n.d.). Method of moments: Negative binomial distribution. Real Statistics Using Excel. Retrieved November 12, 2026, from https://real-statistics.com/distribution-fitting/method-of-moments/method-of-moments-negative-binomial-distribution/